**1. Environment & Configuration Setup**

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: False
GPU count: 0


In [3]:
import os
import torch
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cpu
Configuration loaded.


**LOAD TRAIN FILE**

In [4]:
from huggingface_hub import hf_hub_download

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

train_file_paths = {}

for filename in TRAIN_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    train_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_acts.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_acts.json


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_courts.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_high_courts.json


... (2 outputs left out)

Phase_1_Pretraining/Train/train_supreme_court.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_supreme_court.json


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunals.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_tribunals.json


In [5]:
train_file_paths.items()

dict_items([('Phase_1_Pretraining/Train/train_acts.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json'), ('Phase_1_Pretraining/Train/train_high_courts.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json'), ('Phase_1_Pretraining/Train/train_supreme_court.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json'), ('Phase_1_Pretraining/Train/train_tribunals.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json')])

In [5]:
!pip install -q ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 2.6 MB/s eta 0:00:00a 0:00:01


In [6]:
import ijson
import gc
from pathlib import Path

for filename, path in train_file_paths.items():

    source = Path(filename).stem

    print("Processing:", source)

    count = 0

    with open(path, "rb") as f:
        for record in ijson.items(f, "item"):
            count += 1

    print("Records:", count)

    gc.collect()

Processing: train_acts
Records: 18826
Processing: train_high_courts
Records: 2536738
Processing: train_supreme_court
Records: 55549
Processing: train_tribunals
Records: 869132


In [9]:
import hashlib
import re
import ijson
from pathlib import Path

seen = set()

source_stats = {}

for filename, path in train_file_paths.items():

    source = Path(filename).stem

    unique_records = 0
    unique_chars = 0

    with open(path, "rb") as f:

        for record in ijson.items(f, "item"):

            text = record["text"]

            normalized = re.sub(r"\s+", " ", text).strip()

            key = hashlib.sha256(
                normalized.encode("utf-8")
            ).hexdigest()

            if key in seen:
                continue

            seen.add(key)

            unique_records += 1
            unique_chars += len(normalized)

    source_stats[source] = {
        "unique_records": unique_records,
        "unique_chars": unique_chars
    }

    print(
        source,
        "| unique records:", unique_records,
        "| unique chars:", unique_chars
    )

KeyboardInterrupt: 

In [13]:
del seen

In [7]:
import hashlib
import re
import json
import ijson
from pathlib import Path

for filename, path in train_file_paths.items():

    source = Path(filename).stem
    seen = set()   # reset for this file

    output_path = f"{source}_unique.jsonl"

    total = 0
    unique = 0

    with open(path, "rb") as f, open(output_path, "w", encoding="utf-8") as out:

        for record in ijson.items(f, "item"):

            total += 1
            text = record["text"]

            # Only normalize for duplicate detection
            normalized = re.sub(r"\s+", " ", text).strip()

            h = hashlib.sha256(
                normalized.encode("utf-8")
            ).digest()

            if h in seen:
                continue

            seen.add(h)
            unique += 1

            # Keep original text
            out.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

    print(
        f"{source}: "
        f"{total:,} → {unique:,} unique "
        f"({total-unique:,} duplicates)"
    )

train_acts: 18,826 → 18,732 unique (94 duplicates)
train_supreme_court: 55,549 → 55,362 unique (187 duplicates)
train_tribunals: 869,132 → 864,198 unique (4,934 duplicates)


In [11]:

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

unique_files = [
    "train_acts_unique.jsonl",
    "train_high_courts_unique.jsonl",
    "train_supreme_court_unique.jsonl",
    "train_tribunals_unique.jsonl"
]

source_names = {
    "train_acts_unique.jsonl": "Acts",
    "train_high_courts_unique.jsonl": "High Courts",
    "train_supreme_court_unique.jsonl": "Supreme Court",
    "train_tribunals_unique.jsonl": "Tribunals"
}

# Sample documents for faster analysis
SAMPLE_SIZE = 10000
SEED = 42
rng = np.random.default_rng(SEED)

results = []
length_data = {}

for filename in unique_files:
    source = source_names[filename]

    # Reservoir sampling across the complete source file
    samples = []
    n_seen = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            text = data.get("text", "")

            if not isinstance(text, str) or not text.strip():
                continue

            n_seen += 1

            if len(samples) < SAMPLE_SIZE:
                samples.append(text)
            else:
                j = int(rng.integers(0, n_seen))
                if j < SAMPLE_SIZE:
                    samples[j] = text

    # Tokenize sampled documents using your existing converter
    lengths = []

    for text in samples:
        ids = converter.to_tokens([text])[0]
        lengths.append(len(ids) + 2)  # BOS and EOS

    lengths = np.array(lengths)
    length_data[source] = lengths

    results.append({
        "Source": source,
        "Documents in source": n_seen,
        "Sampled": len(lengths),
        "Mean": round(float(np.mean(lengths)), 2),
        "Median": round(float(np.median(lengths)), 2),
        "P90": round(float(np.percentile(lengths, 90)), 2),
        "P95": round(float(np.percentile(lengths, 95)), 2),
        "Maximum": int(np.max(lengths)),
        "% > 1024": round(float(np.mean(lengths > 1024) * 100), 2)
    })

df = pd.DataFrame(results)
display(df)
df.to_csv("phase1_document_length_statistics.csv", index=False)


,Source,Documents in source,Sampled,Mean,Median,P90,P95,Maximum,% > 1024
0,Acts,18731,10000,8349.50,3409.5,18204.2,29264.35,642685,80.73
1,High Courts,2418248,10000,709.48,384.0,1377.0,2280.10,42528,15.20
2,Supreme Court,55361,10000,4988.32,3072.0,10070.3,14488.00,202667,81.15
3,Tribunals,864198,10000,2168.47,1088.0,4136.8,6701.90,228131,52.18


In [12]:

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

unique_files = {
    "Acts": "train_acts_unique.jsonl",
    "High Courts": "train_high_courts_unique.jsonl",
    "Supreme Court": "train_supreme_court_unique.jsonl",
    "Tribunals": "train_tribunals_unique.jsonl"
}

SAMPLE_SIZE = 10000
SEED = 42

rng = np.random.default_rng(SEED)

# Unicode-aware words containing English letters.
# This is a word-level analysis, separate from model BPE.
word_pattern = re.compile(r"[A-Za-z]+(?:['’-][A-Za-z]+)*")

# Simple function-word list for readable domain terms
stopwords = set("""
a an the and or but if of to in on at by for with from
is are was were be been being have has had do does did
this that these those it its as not no such their there
which who whom whose what where when while into than then
also may shall will would should could can all any each
other same under over between through upon per
""".split())

def extract_words(text):
    return [
        w.lower()
        for w in word_pattern.findall(text)
        if len(w) > 2
    ]

def reservoir_sample(filename, k, rng):
    samples = []
    seen = 0

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            text = obj.get("text", "")

            if not isinstance(text, str) or not text.strip():
                continue

            seen += 1

            if len(samples) < k:
                samples.append(text)
            else:
                j = int(rng.integers(0, seen))
                if j < k:
                    samples[j] = text

    return samples, seen

source_counts = {}
source_totals = {}
sample_details = {}

for source, filename in unique_files.items():
    print("Processing:", source)

    samples, total_docs = reservoir_sample(
        filename, SAMPLE_SIZE, rng
    )

    counter = Counter()

    for text in samples:
        counter.update(extract_words(text))

    source_counts[source] = counter
    source_totals[source] = sum(counter.values())

    sample_details[source] = {
        "documents_in_source": total_docs,
        "sampled_documents": len(samples)
    }

    print("Sampled:", len(samples))
    print("Word occurrences:", source_totals[source])
    print("Unique words:", len(counter))
    print()


Processing: Acts
Sampled: 10000
Word occurrences: 43916686
Unique words: 220955

Processing: High Courts
Sampled: 10000
Word occurrences: 3540731
Unique words: 64481

Processing: Supreme Court
Sampled: 10000
Word occurrences: 27811157
Unique words: 152826

Processing: Tribunals
Sampled: 10000
Word occurrences: 8036462
Unique words: 121567



In [13]:

stats = []

for source, counter in source_counts.items():

    stats.append({
        "Source": source,
        "Sampled documents": sample_details[source]["sampled_documents"],
        "Total words": source_totals[source],
        "Unique words": len(counter),
        "Type-token ratio": round(
            len(counter) / max(1, source_totals[source]), 5
        )
    })

vocab_df = pd.DataFrame(stats)

display(vocab_df)

vocab_df.to_csv(
    "phase1_vocabulary_statistics.csv",
    index=False
)


,Source,Sampled documents,Total words,Unique words,Type-token ratio
0,Acts,10000,43916686,220955,0.00503
1,High Courts,10000,3540731,64481,0.01821
2,Supreme Court,10000,27811157,152826,0.00550
3,Tribunals,10000,8036462,121567,0.01513


In [14]:

top_words_rows = []

for source, counter in source_counts.items():

    filtered = {
        word: count
        for word, count in counter.items()
        if word not in stopwords
    }

    top20 = Counter(filtered).most_common(20)

    print("\n", source)
    print("-" * 40)

    for word, count in top20:
        print(f"{word:25s} {count:,}")

        top_words_rows.append({
            "Source": source,
            "Word": word,
            "Count": count,
            "Per 10k words": round(
                count / source_totals[source] * 10000, 2
            )
        })

top_words_df = pd.DataFrame(top_words_rows)

top_words_df.to_csv(
    "phase1_top20_words.csv",
    index=False
)



 Acts
----------------------------------------
act                       410,615
section                   275,149
government                252,684
state                     194,642
person                    193,948
officer                   166,098
authority                 149,159
order                     145,207
made                      144,787
his                       143,583
case                      141,765
time                      141,535
date                      138,762
provided                  123,857
rules                     123,791
sub-section               122,045
within                    97,307
provisions                96,758
after                     96,218
board                     95,608

 High Courts
----------------------------------------
petitioner                44,868
court                     32,412
learned                   25,888
order                     25,316
case                      23,520
counsel                   20,316
petition               

In [15]:

all_words = set()

for counter in source_counts.values():
    all_words.update(counter.keys())

MIN_COUNT = 30
distinctive_rows = []

for source, counter in source_counts.items():

    other_sources = [
        s for s in source_counts
        if s != source
    ]

    other_counter = Counter()

    for other in other_sources:
        other_counter.update(source_counts[other])

    total_here = source_totals[source]
    total_other = sum(
        source_totals[s] for s in other_sources
    )

    for word, count in counter.items():

        if word in stopwords or count < MIN_COUNT:
            continue

        # Smoothed frequencies per 10,000 words
        freq_here = (
            (count + 0.5) / (total_here + 1)
        ) * 10000

        freq_other = (
            (other_counter[word] + 0.5) /
            (total_other + 1)
        ) * 10000

        ratio = freq_here / freq_other

        distinctive_rows.append({
            "Source": source,
            "Word": word,
            "Count": count,
            "Frequency per 10k": round(freq_here, 3),
            "Relative frequency ratio": round(ratio, 3)
        })

distinctive_df = pd.DataFrame(distinctive_rows)

distinctive_df = distinctive_df.sort_values(
    ["Source", "Relative frequency ratio"],
    ascending=[True, False]
)

for source in source_counts:
    print("\n", source)
    display(
        distinctive_df[
            distinctive_df["Source"] == source
        ].head(15)
    )

distinctive_df.to_csv(
    "phase1_distinctive_words.csv",
    index=False
)



 Acts


,Source,Word,Count,Frequency per 10k,Relative frequency ratio
16950,Acts,comunidades,860,0.196,1543.544
18478,Acts,layout-design,693,0.158,1243.984
18587,Acts,dearnessallowance,576,0.131,1034.112
18585,Acts,familypension,522,0.119,937.248
18775,Acts,gsicc,471,0.107,845.765
5998,Acts,ditto,4088,0.931,814.873
18586,Acts,formaking,441,0.101,791.952
16133,Acts,kuhl,430,0.098,772.221
15552,Acts,canal-officer,412,0.094,739.933
18588,Acts,beless,410,0.093,736.345



 High Courts


,Source,Word,Count,Frequency per 10k,Relative frequency ratio
20214,High Courts,hcservices,1362,3.848,61387.796
19419,High Courts,ablapl,635,1.795,28632.620
20880,High Courts,blapl,306,0.866,13809.438
23566,High Courts,sikshya,128,0.363,5789.601
22699,High Courts,gahc,105,0.298,4753.330
22520,High Courts,wpss,104,0.295,4708.275
21575,High Courts,rongon,99,0.281,4482.999
21204,High Courts,pujahari,90,0.256,4077.501
21762,High Courts,wpms,87,0.247,3942.336
23358,High Courts,magrey,80,0.227,3626.949



 Supreme Court


,Source,Word,Count,Frequency per 10k,Relative frequency ratio
28893,Supreme Court,hon’ble,1221,0.439,4874.718
27748,Supreme Court,citator,905,0.326,3613.637
27230,Supreme Court,pasayat,831,0.299,3318.320
42644,Supreme Court,sivarasan,802,0.289,3202.588
30352,Supreme Court,appellant’s,716,0.258,2859.382
30663,Supreme Court,court’s,670,0.241,2675.807
43568,Supreme Court,mcoca,328,0.118,1310.966
44415,Supreme Court,fhl,307,0.111,1227.160
25443,Supreme Court,headnote,1966,0.707,1121.120
28153,Supreme Court,unders,279,0.100,1115.418



 Tribunals


,Source,Word,Count,Frequency per 10k,Relative frequency ratio
44682,Tribunals,ifjoknh,4325,5.382,27008.086
44619,Tribunals,mihkksdrk,1187,1.478,22243.973
46312,Tribunals,gangnath,1169,1.455,21906.801
49851,Tribunals,qksje,736,0.916,13795.946
50630,Tribunals,mrrjoknh,728,0.906,13646.092
56014,Tribunals,technip,688,0.857,12896.821
46319,Tribunals,cpio's,647,0.806,12128.819
48623,Tribunals,samariya,630,0.785,11810.379
53109,Tribunals,daiuh,578,0.720,10836.327
56013,Tribunals,coflexip,567,0.706,10630.278


In [18]:
unique_files = [
    "train_acts_unique.jsonl",
    "train_high_courts_unique.jsonl",
    "train_supreme_court_unique.jsonl",
    "train_tribunals_unique.jsonl"
]
def sample_texts(files, max_chars_per_file=20_000_000):

    for filename in files:
        chars = 0

        with open(filename, "r", encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                text = record["text"]

                if chars + len(text) > max_chars_per_file:
                    remaining = max_chars_per_file - chars
                    if remaining > 0:
                        yield text[:remaining]
                    break

                yield text
                chars += len(text)


In [27]:
corpus = sample_texts(
    unique_files,
    max_chars_per_file=4_000_000
)

In [ ]:
from pathlib import Path

all_records = {}

for filename, path in train_file_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print(filename, ":", len(records))

    file = Path(filename)
    all_records[file.stem] = records

Phase_1_Pretraining/Train/train_acts.json : 18826


In [7]:
#character count and statics
print("Len of records:",len(records))
print("total characters:",sum(len(data["text"])for data in records))

Len of records: 55549
total characters: 1239650413


In [8]:
print("total characters:",sum(len(data["text"])for data in records[:1000]))

total characters: 36183978


In [9]:
#train and val split
train_text = []
val_text = []

random.seed(SEED)

for data in records[:1000]:
    text = data["text"]

    if random.random() < 0.05:
        val_text.append(text)
    else:
        train_text.append(text)

print(f"Train characters: {sum(len(x) for x in train_text):,}")
print(f"Validation characters: {sum(len(x) for x in val_text):,}")

Train characters: 34,990,401
Validation characters: 1,193,577


**BPE TOKENIZER**

In [8]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list




In [12]:
def train_tokenizer(train_text, vocab_size: int):
   
    num_merges = vocab_size - 261
    bpe = BPETokenizer(num_merges = num_merges)
    vocab = bpe.train(train_text)

    return bpe

**Train BPE TOKENIZER**

In [28]:
corpus = list(corpus)

print("Total chars:",
      sum(len(x) for x in corpus))
start_time = time.time()
bpe = train_tokenizer(corpus, 8000)
elapsed = time.time() - start_time
print("Vocab size:", len(bpe.itos))
print("Training time:", elapsed / 60, "minutes")

Total chars: 16000000
requested merges 7739
Vocab size: 7999
Training time: 8.841635123888652 minutes


In [9]:
import pickle

with open("/kaggle/input/datasets/girieshpassya/tokenizer-results/project_bpe_16k.pkl", "rb") as f:
    bpe = pickle.load(f)

print("BPE saved")

BPE saved


In [23]:

import json
import random
import time
import numpy as np
from pathlib import Path


def collect_tokens(
    unique_files,
    train_budget,
    converter,
    val_tokens=5_000_000,
    val_ratio=0.05,
    seed=42,
    output_dir="hybrid_1b_tokens",
    bos_id=2,
    eos_id=3,
    chunk_size=200_000_000,
):
    total_target = sum(train_budget[f] for f in unique_files)

    if total_target % chunk_size:
        raise ValueError("Total training budget must be divisible by chunk_size.")
    if not 0 < val_ratio < 1:
        raise ValueError("val_ratio must be between 0 and 1.")

    rng = random.Random(seed)
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Remove old outputs so previous partial runs cannot contaminate this run.
    for p in out_dir.glob("train_chunk_*.npy"):
        p.unlink()
    val_path = out_dir / "validation.npy"
    if val_path.exists():
        val_path.unlink()

    n_chunks = total_target // chunk_size
    train_total = 0
    val_total = 0
    chunk_id = 0
    chunk_pos = 0
    train_mm = None
    val_mm = np.lib.format.open_memmap(
        val_path, mode="w+", dtype=np.int32, shape=(val_tokens,)
    )

    # Number of lines consumed in the first pass for each file.
    # The spillover pass skips these lines to prevent record overlap.
    consumed_lines = {f: 0 for f in unique_files}
    per_file_train = {f: 0 for f in unique_files}
    start_time = time.time()

    def open_chunk():
        nonlocal train_mm, chunk_pos, chunk_id

        path = out_dir / f"train_chunk_{chunk_id + 1}.npy"
        train_mm = np.lib.format.open_memmap(
            path, mode="w+", dtype=np.int32, shape=(chunk_size,)
        )
        chunk_pos = 0
        print(f"Writing {path}")

    def append_train(ids):
        nonlocal train_total, chunk_pos, chunk_id, train_mm

        pos = 0
        while pos < len(ids):
            if train_total >= total_target:
                return

            if train_mm is None:
                open_chunk()

            take = min(
                len(ids) - pos,
                chunk_size - chunk_pos,
                total_target - train_total,
            )

            values = np.asarray(ids[pos:pos + take], dtype=np.int32)
            train_mm[chunk_pos:chunk_pos + take] = values

            pos += take
            chunk_pos += take
            train_total += take

            if chunk_pos == chunk_size:
                train_mm.flush()
                del train_mm
                train_mm = None
                chunk_id += 1
                print(
                    f"Completed training chunk {chunk_id}/{n_chunks} "
                    f"({train_total:,} tokens)"
                )

    def add_validation(ids):
        nonlocal val_total

        if val_total >= val_tokens:
            return

        take = min(len(ids), val_tokens - val_total)
        val_mm[val_total:val_total + take] = np.asarray(
            ids[:take], dtype=np.int32
        )
        val_total += take

    # ---------------------------------------------------------
    # PASS 1: Respect the per-corpus budgets.
    # ---------------------------------------------------------
    for filename in unique_files:
        target = train_budget[filename]
        file_train = 0

        print(f"\nPass 1: {filename} | target={target:,}")

        with open(filename, "r", encoding="utf-8") as f:
            for line in f:
                if file_train >= target:
                    break

                consumed_lines[filename] += 1
                data = json.loads(line)
                ids = [bos_id] + converter.to_tokens([data["text"]])[0] + [eos_id]

                # A document is assigned to validation or training, not both.
                if rng.random() < val_ratio and val_total < val_tokens:
                    add_validation(ids)
                else:
                    take = min(len(ids), target - file_train)
                    append_train(ids[:take])
                    file_train += take
                    per_file_train[filename] += take

        print(f"  Training collected: {file_train:,}")

    # ---------------------------------------------------------
    # PASS 2: Fill any shortage using unconsumed records.
    # Priority follows unique_files order (Supreme Court first).
    # ---------------------------------------------------------
    shortage = total_target - train_total

    if shortage > 0:
        print(f"\nRedistributing shortage: {shortage:,} tokens")

        for filename in unique_files:
            if train_total >= total_target:
                break

            extra = 0
            print(f"Pass 2: scanning unused records in {filename}")

            with open(filename, "r", encoding="utf-8") as f:
                # Skip every line already examined in pass 1.
                for _ in range(consumed_lines[filename]):
                    if next(f, None) is None:
                        break

                for line in f:
                    if train_total >= total_target:
                        break

                    data = json.loads(line)
                    ids = [bos_id] + converter.to_tokens([data["text"]])[0] + [eos_id]

                    take = min(len(ids), total_target - train_total)
                    append_train(ids[:take])
                    extra += take
                    per_file_train[filename] += take

            print(f"  Extra training tokens: {extra:,}")

    if train_mm is not None:
        train_mm.flush()
        del train_mm
        train_mm = None

    val_mm.flush()
    del val_mm

    print("\nCollection summary")
    for filename in unique_files:
        print(f"{filename}: {per_file_train[filename]:,} training tokens")

    print(f"Total training:   {train_total:,}")
    print(f"Total validation: {val_total:,}")
    print(f"Elapsed minutes:  {(time.time() - start_time) / 60:.2f}")

    if train_total != total_target:
        raise RuntimeError(
            f"Could collect only {train_total:,}/{total_target:,} training tokens. "
            "The available corpora did not contain enough unused records."
        )

    if val_total != val_tokens:
        raise RuntimeError(
            f"Collected only {val_total:,}/{val_tokens:,} validation tokens. "
            "Increase validation sampling or check corpus availability."
        )

    return {
        "train_tokens": train_total,
        "validation_tokens": val_total,
        "per_file_train": per_file_train,
        "output_dir": str(out_dir),
    }


In [24]:
train_budget = {
    "train_supreme_court_unique.jsonl": 400_000_000,
    "train_high_courts_unique.jsonl": 300_000_000,
    "train_tribunals_unique.jsonl": 200_000_000,
    "train_acts_unique.jsonl": 100_000_000,
}

unique_files = list(train_budget.keys())

result = collect_tokens(
    unique_files=unique_files,
    train_budget=train_budget,
    converter=converter,
    val_tokens=5_000_000,
)


Pass 1: train_supreme_court_unique.jsonl | target=400,000,000
Writing hybrid_1b_tokens/train_chunk_1.npy
Completed training chunk 1/5 (200,000,000 tokens)
Writing hybrid_1b_tokens/train_chunk_2.npy
  Training collected: 272,861,160

Pass 1: train_high_courts_unique.jsonl | target=300,000,000
Completed training chunk 2/5 (400,000,000 tokens)
Writing hybrid_1b_tokens/train_chunk_3.npy
  Training collected: 300,000,000

Pass 1: train_tribunals_unique.jsonl | target=200,000,000
Completed training chunk 3/5 (600,000,000 tokens)
Writing hybrid_1b_tokens/train_chunk_4.npy
  Training collected: 200,000,000

Pass 1: train_acts_unique.jsonl | target=100,000,000
Completed training chunk 4/5 (800,000,000 tokens)
Writing hybrid_1b_tokens/train_chunk_5.npy
  Training collected: 100,000,000

Redistributing shortage: 127,138,840 tokens
Pass 2: scanning unused records in train_supreme_court_unique.jsonl
  Extra training tokens: 0
Pass 2: scanning unused records in train_high_courts_unique.jsonl
Comple

In [15]:
unique_counts = {
    "train_high_courts_unique.jsonl": 2_418_248,
    "train_tribunals_unique.jsonl": 864_198,
    "train_supreme_court_unique.jsonl": 55_361,
    "train_acts_unique.jsonl": 18_732,
}
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
total_records = sum(unique_counts.values())

train_budget = {
    filename: int(TRAIN_TOKENS * count / total_records)
    for filename, count in unique_counts.items()
}

print(train_budget)

{'train_high_courts_unique.jsonl': 144091756, 'train_tribunals_unique.jsonl': 51493398, 'train_supreme_court_unique.jsonl': 3298695, 'train_acts_unique.jsonl': 1116149}


In [10]:
converter = ToTokensConverter(bpe.itos)

In [18]:
import json
import random
import time
unique_files = [
    "train_acts_unique.jsonl",
    "train_high_courts_unique.jsonl",
    "train_supreme_court_unique.jsonl",
    "train_tribunals_unique.jsonl"
]
train_tokens = []
val_tokens = []

VAL_RATIO = 0.05
TARGET_TOKENS = 200_000_000

start_time = time.time()
skipped_train_tokens = 0

for filename in unique_files:
    print(f"Fetching {filename}")

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)

            tokens = converter.to_tokens([data["text"]])[0]
            tokens = [2] + tokens + [3]

            # Keep the validation split separate.
            if random.random() < VAL_RATIO:
                if len(val_tokens) < VAL_TOKENS:
                    remaining = VAL_TOKENS - len(val_tokens)
                    val_tokens.extend(tokens[:remaining])
                continue

            # Skip the first 200M training tokens.
            if skipped_train_tokens < TARGET_TOKENS:
                skip_count = min(
                    len(tokens),
                    TARGET_TOKENS - skipped_train_tokens
                )
                skipped_train_tokens += skip_count
                tokens = tokens[skip_count:]

                if not tokens:
                    continue

            # Collect the next 200M tokens.
            remaining = TARGET_TOKENS - len(train_tokens)
            train_tokens.extend(tokens[:remaining])

            if len(train_tokens) >= TARGET_TOKENS:
                break

    if len(train_tokens) >= TARGET_TOKENS:
        break

print("Training time:", (time.time() - start_time) / 60, "minutes")
print("New train tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))

Fetching train_acts_unique.jsonl
Fetching train_high_courts_unique.jsonl
Training time: 16.999214295546214 minutes
New train tokens: 200000000
Validation tokens: 5000000


In [19]:
import numpy as np

np.save("train_tokens2.npy",
        np.asarray(train_tokens, dtype=np.uint16))

In [86]:
import json
import random
import time

train_tokens = []
val_tokens = []

VAL_RATIO = 0.05

# Build tokenizer converter ONCE

count =0 
start_time = time.time()

for filename in unique_files:

    print(f"Fetching {filename}")

    file_train_target = train_budget[filename]
    file_train_start = len(train_tokens)

    with open(filename, "r", encoding="utf-8") as f:

        for line in f:

            data = json.loads(line)

            # Reuse the same converter
            tokens = converter.to_tokens([data["text"]])[0]
            tokens = [2] + tokens + [3]
            if random.random() < VAL_RATIO:

                if len(val_tokens) < VAL_TOKENS:
                    remaining = VAL_TOKENS - len(val_tokens)
                    val_tokens.extend(tokens[:remaining])

            else:

                current_file_tokens = len(train_tokens) - file_train_start

                if current_file_tokens < file_train_target:
                    remaining = file_train_target - current_file_tokens
                    train_tokens.extend(tokens[:remaining])

            current_file_tokens = len(train_tokens) - file_train_start

            if current_file_tokens >= file_train_target:
                break

    if len(train_tokens) >= TRAIN_TOKENS and count ==0:
        train_tokens = []
        count = 1
    elif len(train_tokens) >= TRAIN_TOKENS:
        break

print("Training time:", (time.time() - start_time) / 60, "minutes")
print("Train tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))

Fetching train_acts_unique.jsonl
Fetching train_high_courts_unique.jsonl
Fetching train_supreme_court_unique.jsonl
Fetching train_tribunals_unique.jsonl
Training time: 6.5986903270085655 minutes
Train tokens: 199999998
Validation tokens: 5000000


In [88]:
import numpy as np

np.save("train_tokens1.npy",
        np.asarray(train_tokens, dtype=np.uint16))

np.save("val_tokens1.npy",
        np.asarray(val_tokens, dtype=np.uint16))